# Orientation: Kalman filter

## Overview

The Kalman filter is a widely used estimation algorithm that incorporates a deterministic model of the system, its stochastic/probabilistic characteristics and noisy measurements to estimate a set of state variables over time.  

It has the following components:

- *State vector*: set of parameters that describe the system, which are estimated by the algorithm.
- *Measurement vector*: set of observed measurements taken simultaneously from the system.  
- *System model*: describes how the state vector and the error covariance matrix evolve over time. It's deterministic.
- *Measurement model*: describes how measurements are a function of the state vector. It's deterministic.
- *Noise covariance matrices*: capture randomness on both the system and measurements.
- *Error covariance matrix*: represents the uncertainty in the state vector estimate, and the correlation between the different state variables.

There are a lot of variants of the filter, namely: classical Kalman filter, additive and multiplicative extended Kalman filters (EKF/AEKF and MEKF), invariant Kalman filter (IKF), error-state Kalman filter (ESKF), unscented Kalman filter (UKF), multiple hypothesis Kalman filter (MHKF), Fast Kalman filter (FKF), etc. It's also closely related to particle filters and other Bayesian filtering techniques. At the same time, one type of filter can vary in its implementation details: order of aproximations, adaptive/fixed covariance matrices, closed-loop/open-loop, sequential/vector measurement update, white/time-correlated noise, etc. 

For the simplest case, the classical Kalman filter, it works under the following assumptions. First, the system dynamics are described by:
$$
\mathbf{\dot{x}}(t) = \mathbf{F}(t) \mathbf{x}(t) + \mathbf{G}(t) \mathbf{w}_s(t)
$$
where:
- $\mathbf{x}(t)$ is the state vector,
- $\mathbf{F}(t)$ is the system matrix,
- $\mathbf{G}(t)$ is the system noise distribution matrix,
- $\mathbf{w}_s(t)$ is the system noise vector, which comprises many independent random noise sources, each assumed to have a zero-mean symmetric distribution, such as the normal distribution.

Secondly, the measurements are described by:
$$
\mathbf{z}(t) = \mathbf{H}(t) \mathbf{x}(t) + \mathbf{w}_m(t)
$$
where:
- $\mathbf{z}(t)$ is the measurement vector,
- $\mathbf{H}(t)$ is the measurement matrix,
- $\mathbf{w}_m(t)$ is the measurement noise vector, with the same assumptions as the system noise vector.

## Basic algorithm

On a high level, it operates as follows. First, the state is predicted forward from the immediate past using the system model. This, in turn, increases uncertainty. This can be imagined as a normal distribution that's moved and stretched across the space. Secondly, information from the measurements (sensor outputs) is brought in as another distribution and compared against the prediction, which is done by mapping from state space to measurement space through the measurement model and taking the difference between the predicted and actual measurements, known as the innovation. To decide which one to trust more, a variable gain is computed, the Kalman gain, which solves the optimization problem of finding a third distribution that minimizes the covariances. Finally, this gain is used to update both state and the uncertainty. Further visual explanations for this process can be found at [2] and [3].     

This translates into the following steps, as taken from [1]. For each time step $k$, do:

*System propagation*:

1. Calculate the transition matrix, $\mathbf{\Phi}_{k-1}$.
2. Calculate the system noise covariance matrix, $\mathbf{Q}_{k–1}$.
3. Propagate the state vector estimate from $\mathbf{\hat{x}}_{k−1}^+$ to $\mathbf{\hat{x}}_k^−$: 
$$
\mathbf{\hat{x}}_k^{-} = \mathbf{\Phi}_{k-1} \mathbf{\hat{x}}_{k-1}^{+}
$$
4. Propagate the error covariance matrix from $\mathbf{P}_{k–1}^+$ to $\mathbf{P}_k^−$:
$$
\mathbf{P}_k^{-} = \mathbf{\Phi}_{k-1} \mathbf{P}_{k-1}^+ \mathbf{\Phi}_{k-1}^T + \mathbf{Q}_{k-1}
$$

*Measurement update*:

5. Calculate the measurement matrix, $\mathbf{H_k}$.
6. Calculate the measurement noise covariance matrix, $\mathbf{R}_k$.
7. Calculate the Kalman gain matrix, $\mathbf{K}_k$:
$$
\mathbf{K}_k = \mathbf{P}_k^{-} \mathbf{H}_k^T (\mathbf{H}_k \mathbf{P}_k^{-} \mathbf{H}_k^T + \mathbf{R}_k)^{-1}
$$
8. Formulate the measurement, $\mathbf{z}_k$.
9. Update the state vector estimate from $\mathbf{\hat{x}}_k^−$ to $\mathbf{\hat{x}}_k^+$:
$$
\mathbf{\hat{x}}_k^{+} = \mathbf{\hat{x}}_k^{-} + \mathbf{K}_k (\mathbf{\mathbf{z}}_k - \mathbf{H}_k \mathbf{\hat{x}}_k^{-})
= \mathbf{\hat{x}}_k^{-} + \mathbf{K}_k \boldsymbol\delta \mathbf{z}_k^{-}
$$
10. Update the error covariance matrix from $\mathbf{P}_k^−$ to $\mathbf{P}_k^+$:
$$
\mathbf{P}_k^{+} = (\mathbf{I} - \mathbf{K}_k \mathbf{H}_k) \mathbf{P}_k^{-}
$$
Alternatively (Joseph form, more numerically stable):
$$
\mathbf{P}_k^{+} = (\mathbf{I} - \mathbf{K}_k \mathbf{H}_k) \mathbf{P}_k^{-} (\mathbf{I} - \mathbf{K}_k \mathbf{H}_k)^T + \mathbf{K}_k \mathbf{R}_k \mathbf{K}_k^T
$$

The matrices $\mathbf{\Phi}$, $\mathbf{Q}$, $\mathbf{H}$, and $\mathbf{R}$ need to be defined/derived by the user depending on the application, as well as the initial conditions, $\hat{\mathbf{x}}_0^+$ and $\mathbf{P}_0^+$. For the case of the transition matrix, it can be obtained from the system matrix with a matrix exponential:
$$
\mathbf{\Phi}_{k-1} = e^{\mathbf{F}_{k-1} \Delta t} = \sum_{n=0}^{\infty} \frac{(\mathbf{F}_{k-1} \Delta t)^n}{n!} = \mathbf{I} + \Delta t \mathbf{F}_{k-1} + \frac{\Delta t ^2}{2} \mathbf{F}_{k-1}^2 + \frac{\Delta t ^3}{6} \mathbf{F}_{k-1}^3 + \dots
$$
For the case of the noise covariance matrices, they can be, on the simplest case, diagonal matrices with covariance values based on sensor specifications or empirical observations.

A diagram of the dependencies between the variables is shown below.

<img src="./assets/kf-dependencies.png" alt="Kalman filter dependencies" width="400" height="350">


## Application: quaternion-based orientation estimation

TODO: think what do with gyro biases

### EKF general formulation

The assumption of linear system and measurement models may not hold well for this application, which motivates the use of the EKF, one of the simplest non-linear alternatives. The EKF linearizes the models through the Jacobian matrix around the current estimate and then feeds them into the standard Kalman filter equations. This means that now the system dynamic model and measurement model are:
$$
\mathbf{\dot{x}}(t) =  \mathbf{f}(\mathbf{x}(t), \mathbf{u}(t), t) + \mathbf{G}(t) \mathbf{w}_s(t) \\
\mathbf{z}(t) = \mathbf{h}(\mathbf{x}(t), t) + \mathbf{w}_m(t)
$$
Where $\mathbf{u}(t)$ is an additional control input to the system.
They are linearized as follows:
$$
\mathbf{F}_{k-1} = \left .\frac{\partial \mathbf{f}}{\partial \mathbf{x}}\right|_{\mathbf{x} = \hat{\mathbf{x}}_{k-1}^{+}, \mathbf{u} = \mathbf{u}_{k}, t = t_k} \\
\mathbf{H}_k = \left .\frac{\partial \mathbf{h}}{\partial \mathbf{x}}\right|_{\mathbf{x} = \hat{\mathbf{x}}_k^{-}, t = t_k}
$$
The derivations that follow were aided by the documentation from [5].

### Transition matrix

Since quaternions have already been explored for the Mahony filter and are a suitable representation for orientation, they will be used inside the state vector. The quaternion differential equation that describes the evolution of the quaternion can be expressed as:
$$
\dot{\mathbf{q}} = \frac{1}{2} \mathbf{q}_{b}^{w} \otimes (0, \bm{\omega}^{b}) = \frac{1}{2} \boldsymbol\Omega (\boldsymbol\omega)\mathbf{q}_{b}^{w} = 
\frac{1}{2} \begin{bmatrix}
0 & -\omega_x & -\omega_y & -\omega_z \\
\omega_x & 0 & \omega_z & -\omega_y \\
\omega_y & -\omega_z & 0 & \omega_x \\
\omega_z & \omega_y & -\omega_x & 0
\end{bmatrix} \begin{bmatrix}q_0 \\ q_1 \\ q_2 \\ q_3\end{bmatrix} = 
\begin{bmatrix}
-\frac{1}{2} \omega_x q_1 - \frac{1}{2} \omega_y q_2 - \frac{1}{2} \omega_z q_3 \\
\frac{1}{2} \omega_x q_0 + \frac{1}{2} \omega_z q_2 - \frac{1}{2} \omega_y q_3 \\
\frac{1}{2} \omega_y q_0 - \frac{1}{2} \omega_z q_1 + \frac{1}{2} \omega_x q_3 \\
\frac{1}{2} \omega_z q_0 + \frac{1}{2} \omega_y q_1 - \frac{1}{2} \omega_x q_2
\end{bmatrix}
$$
And can be viewed as the continuous-time system model, $\mathbf{f}(\mathbf{q}_{b}^{w}(t), \bm{\omega}^{b}(t), t)$. Assuming from now on that the quaternion maps from the body frame to the world/reference frame to drop the subscript and superscript, the Jacobian can be computed as:
$$
\mathbf{F} = \frac{\partial \mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial \mathbf{q}} = \begin{bmatrix}
\frac{\partial\mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial q_0} &
\frac{\partial\mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial q_1} &
\frac{\partial\mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial q_2} &
\frac{\partial\mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial q_3}
\end{bmatrix} = \begin{bmatrix}
0 & -\frac{1}{2} \omega_x & -\frac{1}{2} \omega_y & -\frac{1}{2} \omega_z  \\
\frac{1}{2} \omega_x & 0 & \frac{1}{2} \omega_z & -\frac{1}{2} \omega_y \\
\frac{1}{2} \omega_y & -\frac{1}{2} \omega_z & 0 & \frac{1}{2} \omega_x \\
\frac{1}{2} \omega_z & \frac{1}{2} \omega_y & -\frac{1}{2} \omega_x & 0
\end{bmatrix}
$$
Lastly, with a first order approximation, the transition matrix can be expressed as:
$$
\mathbf{\Phi}_{k-1} = e^{\mathbf{F}_{k-1} \Delta t} \approx \mathbf{I}_4 + \Delta t \mathbf{F}_{k-1} = \begin{bmatrix}
1 & -\frac{\Delta t}{2} \omega_x & -\frac{\Delta t}{2} \omega_y & -\frac{\Delta t}{2} \omega_z \\
\frac{\Delta t}{2} \omega_x & 1 & \frac{\Delta t}{2} \omega_z & -\frac{\Delta t}{2} \omega_y \\
\frac{\Delta t}{2} \omega_y & -\frac{\Delta t}{2} \omega_z & 1 & \frac{\Delta t}{2} \omega_x \\
\frac{\Delta t}{2} \omega_z & \frac{\Delta t}{2} \omega_y & -\frac{\Delta t}{2} \omega_x & 1
\end{bmatrix}
$$
It's assumed that angular velocities had their stationary bias removed before being fed into the filter. 

### Measurement matrix

Accelerometer readings can be fed into the measurements of the algorithm. When the device is approximately stationary, the specific force must closely match the inverse of the gravity vector in the body frame. When the device is moving, although the filter should asign a smaller gain to the measurements, the correction phase could be be skipped to save computational resources, but this needs to be further tested. To map from the state space to the measurement space, the matrix representation of quaternion can be used, as it was shown in the "Rotations in 3D space: Quaternions - Conversion to other representations" section of the [Manohy filter notebook](2-mahonyQuaternion.ipynb):
$$
\bm{f}^b = -\mathbf{g}^b = -\mathbf{C}_{w}^{b} \mathbf{g}^w = -(\mathbf{C}_{b}^{w})^{-1} \mathbf{g}^w = -(\mathbf{C}_{b}^{w})^{T} \mathbf{g}^w = \\
-\begin{bmatrix}
q_0^2 + q_1^2 - q_2^2 - q_3^2 & 2(q_1 q_2 - q_0 q_3) & 2(q_1 q_3 + q_0 q_2) \\
2(q_1 q_2 + q_0 q_3) & q_0^2 - q_1^2 + q_2^2 - q_3^2 & 2(q_2 q_3 - q_0 q_1) \\
2(q_1 q_3 - q_0 q_2) & 2(q_2 q_3 + q_0 q_1) & q_0^2 - q_1^2 - q_2^2 + q_3^2
\end{bmatrix}^T \begin{bmatrix}0 \\ 0 \\ -g \end{bmatrix} = 
\begin{bmatrix}
q_0^2 + q_1^2 - q_2^2 - q_3^2 & 2(q_1 q_2 + q_0 q_3) & 2(q_1 q_3 - q_0 q_2) \\
2(q_1 q_2 - q_0 q_3) & q_0^2 - q_1^2 + q_2^2 - q_3^2 & 2(q_2 q_3 + q_0 q_1) \\
2(q_1 q_3 + q_0 q_2) & 2(q_2 q_3 - q_0 q_1) & q_0^2 - q_1^2 - q_2^2 + q_3^2
\end{bmatrix} \begin{bmatrix}0 \\ 0 \\ g \end{bmatrix} = 
\begin{bmatrix}
2g (q_1 q_3 - q_0 q_2) \\
2g (q_2 q_3 + q_0 q_1) \\
g (q_0^2 - q_1^2 - q_2^2 + q_3^2)
\end{bmatrix} = \mathbf{h}(\mathbf{q}(t), t)
$$
Computing the Jacobian yields the measurement matrix:
$$
\mathbf{H}_k = \frac{\partial \mathbf{h}(\mathbf{q})}{\partial \mathbf{q}} = 2g \begin{bmatrix}
-q_2 & q_3 & -q_0 & q_1 \\
q_1 & q_0 & q_3 & q_2 \\
q_0 & -q_1 & -q_2 & q_3 \\
\end{bmatrix}
$$

### System noise covariance matrix

Randomness in the system model is primarily due to the noise in the gyroscope measurements. A covariance matrix of bias-free angular velocities can be calculated from stationary measurements as follows:
$$
\mathbf{\Sigma}_\omega = \text{cov}(\bm{\omega}) = \begin{bmatrix}
\text{var}(\omega_x) & \text{cov}(\omega_x, \omega_y) & \text{cov}(\omega_x, \omega_z) \\
\text{cov}(\omega_x, \omega_y) & \text{var}(\omega_y) & \text{cov}(\omega_y, \omega_z) \\
\text{cov}(\omega_x, \omega_z) & \text{cov}(\omega_y, \omega_z) & \text{var}(\omega_z) 
\end{bmatrix} = \begin{bmatrix}
\sigma_{\omega_x}^2 & r_{\omega_x \omega_y} \sigma_{\omega_x} \sigma_{\omega_y} & r_{\omega_x \omega_z} \sigma_{\omega_x} \sigma_{\omega_z} \\
r_{\omega_x \omega_y} \sigma_{\omega_x} \sigma_{\omega_y} & \sigma_{\omega_y}^2 & r_{\omega_y \omega_z} \sigma_{\omega_y} \sigma_{\omega_z} \\
r_{\omega_x \omega_z} \sigma_{\omega_x} \sigma_{\omega_z} & r_{\omega_y \omega_z} \sigma_{\omega_y} \sigma_{\omega_z} & \sigma_{\omega_z}^2
\end{bmatrix}
$$
Where $r_{\omega_i\omega_j}$ is the correlation coefficient between $\omega_i$ and $\omega_j$, and $\sigma_{\omega_i}$ is the standard deviation of $\omega_i$. This matrix has to be mapped from angular velocity space to the quaternion space. Holding angular velocity constant, a small change in it due to noise, $\delta \boldsymbol\omega$, will induce a change in the quaternion, $\delta \mathbf{q}_k$ over the time step $\Delta t$. Estimating the quaternion at the end of the interval with a first order approximation gives:
$$
\mathbf{q}_k = \mathbf{q}_{k-1} + \Delta t \dot{\mathbf{q}}_{k-1} =
\mathbf{q}_{k-1} + \Delta t \mathbf{f}(\mathbf{q}_{k-1}, \boldsymbol\omega_{k-1}) \\
$$
Using a Taylor series expansion, differentiating with respect to the angular velocity, yields:
$$
\delta \mathbf{q}_k = \frac{\Delta t}{1!} \frac{\partial \mathbf{f}}{\partial \boldsymbol\omega} \delta \boldsymbol\omega + \frac{\Delta t}{2!} \frac{\partial^2 \mathbf{f}}{\partial \boldsymbol\omega^2} (\delta \boldsymbol\omega)^2 + \frac{\Delta t}{3!} \frac{\partial^3 \mathbf{f}}{\partial \boldsymbol\omega^3} (\delta \boldsymbol\omega)^3 + \dots
$$
Using a first order approximation:
$$
\delta \mathbf{q}_k \approx \Delta t \frac{\partial \mathbf{f}}{\partial \boldsymbol\omega} \delta \boldsymbol\omega = \mathbf{W}_k \delta \boldsymbol\omega, \quad \mathbf{W}_k = \frac{\Delta t}{2} \begin{bmatrix}
-q_1 & -q_2 & -q_3 \\
q_0 & -q_3 & q_2 \\
q_3 & q_0 & -q_1 \\
-q_2 & q_1 & q_0
\end{bmatrix}
$$
This new Jacobian matrix $W_k$ can be viewed as a linear transformation that captures the sensitivity of the quaternion with respect to the angular velocity. To obtain the system noise covariance matrix, as shown in [6] the following relationship holds for the covariance mapping:
$$
\mathbf{Q}_{k} = \mathbf{W}_k \mathbf{\Sigma}_\omega \mathbf{W}_k^T
$$

### Measurement noise covariance matrix

This matrix includes noise from the accelerometer measurements. First, compute the stationary residuals of specific force measurements from each capture and aggregate the results:
$$
\bm{f}' = \bm{f} - \bm{\bar{f}} \approx \bm{f} - \mathbf{g}
$$
Secondly, to account for interactions between different axes, compute the full covariance matrix of the stationary residuals:
$$
\mathbf{R}_{k} = \mathbf{\Sigma}_{f} = \text{cov}(\bm{f}') = \begin{bmatrix}
\text{var}(f'_x) & \text{cov}(f'_x, f'_y) & \text{cov}(f'_x, f'_z) \\
\text{cov}(f'_x, f'_y) & \text{var}(f'_y) & \text{cov}(f'_y, f'_z) \\
\text{cov}(f'_x, f'_z) & \text{cov}(f'_y, f'_z) & \text{var}(f'_z)
\end{bmatrix} = \begin{bmatrix}
\sigma_{f'_x}^2 & r_{f_x f_y} \sigma_{f'_x} \sigma_{f'_y} & r_{f_x f_z} \sigma_{f'_x} \sigma_{f'_z} \\
r_{f'_x f'_y} \sigma_{f'_x} \sigma_{f'_y} & \sigma_{f'_y}^2 & r_{f_y f_z} \sigma_{f'_y} \sigma_{f'_z} \\
r_{f'_x f'_z} \sigma_{f'_x} \sigma_{f'_z} & r_{f'_y f'_z} \sigma_{f'_y} \sigma_{f'_z} & \sigma_{f'_z}^2
\end{bmatrix}
$$

### Other considerations

* If the stationary residuals of angular velocity and specific force measurements seem to be uncorrelated (i.e. $r \approx 0$), the noise covariance matrices can be simplified to:
$$
\mathbf{\Sigma}_\omega = \begin{bmatrix}
\sigma_{\omega_x}^2 & 0 & 0 \\
0 & \sigma_{\omega_y}^2 & 0 \\
0 & 0 & \sigma_{\omega_z}^2
\end{bmatrix}, 
\mathbf{\Sigma}_f = \begin{bmatrix}
\sigma_{f'_x}^2 & 0 & 0 \\
0 & \sigma_{f'_y}^2 & 0 \\
0 & 0 & \sigma_{f'_z}^2
\end{bmatrix}
$$

* The EKF variant discussed above doesn't guarantee that the quaternion will remain normalized and thus a valid rotation. This limitation is introduced by the fact that corrections are applied additively to the quaternion state instead of multiplicatively. Therefore, it must be included within step 9 of the algorithm to normalize the quaternion:
$$
\hat{\mathbf{q}}_k^+ \leftarrow \frac{1}{\|\hat{\mathbf{q}}_k^+\|}\hat{\mathbf{q}}_k^+
$$
The error covariance matrix also has to be updated, otherwise it will become inconsistent with the normalized quaternion state. First define a normalization function:
$$
\hat{\mathbf{q}} = \mathbf{n}(\mathbf{q}) = \frac{\mathbf{q}}{\|\mathbf{q}\|}
$$
Second, compute its Jacobian with respect to the quaternion around the current estimate:
$$
\mathbf{J} = \frac{\partial \mathbf{n}}{\partial \mathbf{q}} = \frac{1}{\|\mathbf{q}\|} \left( \mathbf{I}_4 - \frac{1}{\|\mathbf{q}\|^2} \mathbf{N}  \right), 
\quad \mathbf{N} = \begin{bmatrix}
q_0^2 & q_0 q_1 & q_0 q_2 & q_0 q_3 \\
q_1 q_0 & q_1^2 & q_1 q_2 & q_1 q_3 \\
q_2 q_0 & q_2 q_1 & q_2^2 & q_2 q_3 \\
q_3 q_0 & q_3 q_1 & q_3 q_2 & q_3^2
\end{bmatrix}
$$
Lastly correct the error covariance matrix with the Jacobian as follows:
$$
\mathbf{P}_k^+ \leftarrow \mathbf{J} \mathbf{P}_k^+ \mathbf{J}^{T}
$$
If the filter fails despite applying these corrections, more specialized techniques such as the IKF, MEKF or ESKF must be considered.

* Compute the world/reference frame and the initial orientation $\hat{\mathbf{q}}_0^+$ as it was done on the Mahony filter. 

* The initial error covariance matrix $\mathbf{P}_0^+$ can be computed by treating of a set of quaternions as 4D-vectors and taking the covariance on the residuals obtained from substracting the initial orientation. First, compute the quaternions from disjoint subblocks of consecutive samples during the initialization period, proceeding as with the initial orientation, making sure each one has roughly the same number of samples. Second, since for every orientation there exists two quaternions representing the same rotation (i.e., $\mathbf{q}$ and $-\mathbf{q}$), choose the one that is closest to the initial orientation:
$$
\hat{\mathbf{q}}_i \leftarrow \text{sign}(\hat{\mathbf{q}}_0^+ \cdot \hat{\mathbf{q}}_i) \hat{\mathbf{q}}_i
$$
Third, compute the covariance of the residuals:
$$
\mathbf{q}_i' = \hat{\mathbf{q}}_i - \hat{\mathbf{q}}_0^+ \\
\mathbf{P}_0^+ \approx \text{cov}(\mathbf{q}') = \begin{bmatrix}
\text{var}(q_0') & \text{cov}(q_0', q_1') & \text{cov}(q_0', q_2') & \text{cov}(q_0', q_3') \\
\text{cov}(q_0', q_1') & \text{var}(q_1') & \text{cov}(q_1', q_2') & \text{cov}(q_1', q_3') \\
\text{cov}(q_0', q_2') & \text{cov}(q_1', q_2') & \text{var}(q_2') & \text{cov}(q_2', q_3') \\
\text{cov}(q_0', q_3') & \text{cov}(q_1', q_3') & \text{cov}(q_2', q_3') & \text{var}(q_3')
\end{bmatrix}
$$
Note that this approach is not perfect, as "deviations" are computed as if quaternions lived on regular Euclidean space (they do not), the matrix is rank-deficient (4x4 but only 3 degrees of freedom) and uncertainty may be overestimated (higher noise levels on the subblocks' orientations compared to the general estimate). However, more precise methods (related to the more specialized filters mentioned earlier) seem much more involved. On the other hand, this method is probably better than just using a diagonal matrix with arbitrary values.

## Sources:

1. [Principles of GNSS, Inertial, and Multisensor Integrated Navigation Systems - Paul D. Groves - Chapter 3](../../docs/assets/books/Principles%20of%20GNSS,%20Inertial,%20and%20Multisensor%20Integrated%20Navigation%20Systems%20-%20Paul%20Groves.pdf)
2. [Why Kalman Filters Work So Well](https://www.youtube.com/watch?v=zFmO_b05jU4)
3. [Kalman Filter - VISUALLY EXPLAINED!](https://www.youtube.com/watch?v=-DiZGpAh7T4)
4. [Wikipedia: Matrix Exponential](https://en.wikipedia.org/wiki/Matrix_exponential)
5. [AHRS: Extended Kalman Filter Documentation](https://ahrs.readthedocs.io/en/latest/filters/ekf.html#)
6. [Wikipedia: Covariance](https://en.wikipedia.org/wiki/Covariance#Auto-covariance_matrix_of_real_random_vectors)
7. [AHRS: Fast Kalman Filter Documentation](https://ahrs.readthedocs.io/en/latest/filters/fkf.html#)
8. [AHRS: Unscented Kalman Filter Documentation](https://ahrs.readthedocs.io/en/latest/filters/ukf.html#)
9. [A Comparative Analysis Between the Additive and
the Multiplicative Extended Kalman Filter for
Satellite Attitude Determination - Hassan et al.](../../docs/assets/papers/A%20Comparative%20Analysis%20Between%20the%20Additive%20and%20the%20Multiplicative%20Extended%20Kalman%20Filter%20for%20Satellite%20Attitude%20Determination%20-%20Hassan%20et%20al.pdf)
10. [Multiplicative vs. Additive Filtering for Spacecraft Attitude Determination - Markley](../../docs/assets/papers/Multiplicative%20vs.%20Additive%20Filtering%20for%20Spacecraft%20Attitude%20Determination%20-%20Markley.pdf)
11. [Invariant Kalman Filter Overview](https://www.emergentmind.com/topics/invariant-kalman-filter)
12. [https://www.emergentmind.com/topics/multiplicative-extended-kalman-filter-mekf](https://www.emergentmind.com/topics/multiplicative-extended-kalman-filter-mekf)
13. [Error-State Kalman Filter (ESKF)](https://www.emergentmind.com/topics/error-state-kalman-filter-eskf)